In [1]:
# Quarterly depedency graph

In [1]:
import os
os.getcwd()

'c:\\Users\\giovanni.sgaravatti\\Bruegel Gitlab\\2021-11-european-natural-gas-imports\\standalone pieces of code'

In [2]:
Share_point = r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Data' # Gio
os.chdir(Share_point)

In [3]:
import json
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.dates import DateFormatter
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns

In [4]:
# import ENTSOG pipeline data
entsog = pd.read_csv(r'Imports\EU27\df1.csv')
del entsog['dates.1']
entsog = entsog.set_index(pd.DatetimeIndex(entsog['dates']))
del entsog['dates']

In [5]:
# import LNG data from GIE (we only know where the LNG arrives, not where it comes from)
agsi = pd.read_csv(r'C:\\Users\\giovanni.sgaravatti\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals\\raw_data\\agsi.csv') # Gio
agsi=agsi.set_index(pd.DatetimeIndex(agsi['dates']))
del agsi['dates']
del agsi['index']

In [6]:
# import Bloomberg LNG data (with these data we know both where it comes from and where it arrives, but we trust GIE better - also to be consistent with the tracker)
lng_b = pd.read_excel(r'LNG\Bloomberg\granular LNG imports.xlsx') # Gio
lng_b.rename(columns= {'Unnamed: 0':'dates'},inplace=True)
lng_b = lng_b.set_index(pd.DatetimeIndex(lng_b['dates']))
del lng_b['dates']

In [7]:
lng_b['Tot'] = lng_b.sum(axis=1,numeric_only=True)

In [8]:
# Divide each column by the 'Tot' column
ratios_df = lng_b.div(lng_b['Tot'], axis=0)

In [17]:
agsi_m = agsi.groupby(pd.Grouper(freq='M'))['sendOut'].sum(numeric_only=True)
# take only values after 2019 to be consistent with Bloomberg data
agsi_19 = agsi_m['2019':]

In [18]:
# to align with Bloombgerg lng
agsi_19= agsi_19.iloc[:-2]

In [20]:
# create new dataframe
lng = pd.DataFrame()
lng['dates'] = agsi_19.index

In [21]:
 # multiply Bloomberg LNG ratios by AGSI totals
# and convert to M3m
for column in ratios_df.columns:
    lng[column] = agsi_19.values*ratios_df[column].values/10.3

In [22]:
lng.set_index(pd.DatetimeIndex(lng['dates']),inplace=True)
del lng['dates']

In [23]:
lng['Total less Russia and USA'] = lng['Tot'] - lng['Russia'] - lng['United States']

In [24]:
# Change the months for which you have data here
months = pd.date_range(start='2019-01-01', end='2023-10-31', freq='M')

In [25]:
entsog = entsog['2019':]

In [26]:
converter = 10300000   ## KWh to M3m --> 10.3 KWh/m^3     # on ENTSOG/AGSI the data comes in KWh, we transform it (later on) in M3m 

In [27]:
entsog_m = pd.DataFrame()
entsog_m['dates'] = months
entsog_m.set_index(pd.DatetimeIndex(entsog_m['dates']),inplace=True)
del entsog_m['dates']

for country in ['Russia', 'Norway','Algeria', 'UK', 'Azerbaijan','Libya']:
    entsog_m[country] = entsog[entsog['aggregation'] == country]['values'].groupby(pd.Grouper(freq='M')).sum(numeric_only=True)/converter

In [28]:
for pipe in ['Ukraine Gas Transit', 'Yamal (BY,PL)','Nord Stream', 'Turkstream']:
    entsog_m[pipe] = entsog[entsog['aggregation2'] == pipe]['values'].groupby(pd.Grouper(freq='M')).sum(numeric_only=True)/converter

In [29]:
entsog_q = entsog_m.groupby(pd.Grouper(freq='Q')).sum(numeric_only=True)
lng_q = lng.groupby(pd.Grouper(freq='Q')).sum(numeric_only=True)

In [31]:
graph = entsog_q
graph['USA LNG'] = lng_q['United States']
graph['Russia LNG'] = lng_q['Russia']
del graph['Russia']
graph['LNG less RU and USA'] = lng_q['Total less Russia and USA']
graph = graph['2021':]

In [34]:
graph.tail()

,Nord Stream,"Yamal (BY,PL)",Ukraine Gas Transit,Turkstream,Russia LNG,Libya,Norway,Algeria,Azerbaijan,UK,LNG less RU and USA,USA LNG
dates,,,,,,,,,,,,
2022-12-31,0.0,0.0,3694.221499,3147.891807,4624.932399,873.443851,23426.364254,8730.887417,3307.228577,6400.859889,17273.915748,12597.258649
2023-03-31,0.0,0.0,2837.380097,2637.472494,5025.347501,691.297353,23450.228969,7318.670448,3068.134926,4850.609673,13134.527503,14147.804607
2023-06-30,0.0,0.0,3267.948682,2588.875575,4601.882065,722.073983,22424.073815,8509.097454,3037.083973,6484.323957,14552.060877,16792.600747
2023-09-30,0.0,0.0,3248.204904,4388.088996,3885.493073,506.069164,20593.586580,8922.464988,3050.647476,3562.232558,12697.793489,14435.160040
2023-12-31,0.0,0.0,1097.735271,1444.347627,NaN,195.823158,7795.722842,2799.485356,1143.079442,1047.242622,NaN,NaN


In [33]:
graph = graph[['Nord Stream','Yamal (BY,PL)','Ukraine Gas Transit','Turkstream','Russia LNG', 'USA LNG','LNG less RU and USA', 'Norway','Algeria','UK','Azerbaijan','Libya']]

In [35]:
from datetime import datetime
today = date.today()

In [37]:
with pd.ExcelWriter("Other\quarterly_data {}.xlsx".format(today)) as writer:
    Excelwriter = pd.ExcelWriter("Other\quarterly_data {}.xlsx".format(today),engine="xlsxwriter")
    entsog_q.to_excel(Excelwriter, sheet_name="ENTSOG", index=True)
    lng_q.to_excel(Excelwriter, sheet_name="LNG", index=True)
    graph.to_excel(Excelwriter, sheet_name="graph", index=True)
Excelwriter.close()
Excelwriter.save()

C:\Users\giovanni.sgaravatti\AppData\Local\Temp\ipykernel_29576\2207928906.py:7: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  Excelwriter.save()
c:\Users\giovanni.sgaravatti\anaconda3\lib\site-packages\xlsxwriter\workbook.py:339: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")
